<a href="https://colab.research.google.com/github/keshavarzIT/AI-Agenting/blob/main/marketingAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip uninstall google-generativeai
!pip install google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Would remove:
    /usr/local/lib/python3.13/dist-packages/google/generativeai/*
    /usr/local/lib/python3.13/dist-packages/google_generativeai-0.8.6-py3.13-nspkg.pth
    /usr/local/lib/python3.13/dist-packages/google_generativeai-0.8.6.dist-info/*
Proceed (Y/n)? Y
  Successfully uninstalled google-generativeai-0.8.6


In [ ]:
import time
import requests
import pandas as pd
import json
from google import genai
from google.colab import userdata
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

GEMINI_KEY = userdata.get("marketingAgent")
NESHAN_KEY = userdata.get("NESHAN_API_KEY")
# ۱. تنظیمات کلیدها
NESHAN_API_KEY = NESHAN_KEY  # کلید API نشان از platform.neshan.org
GEMINI_API_KEY = GEMINI_KEY     # کلید جمنای از Google AI Studio

client = genai.Client(api_key=GEMINI_API_KEY)

# تنظیم سشن با قابلیت تلاش مجدد برای جلوگیری از خطای Timeout
session = requests.Session()
retries = Retry(total=3, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# ۲. فهرست کامل مناطق و محدوده های جغرافیایی شیراز (پوشش کامل ۱۱ منطقه شهرداری + شهرک‌ها)
SHIRAZ_ZONES = [
    {"name": "منطقه ۱ (معالی‌آباد، فرهنگ‌شهر، قصردشت)", "lat": 29.6732, "lng": 52.4735},
    {"name": "منطقه ۲ (دروازه اصفهان، فلکه هنگ، خیابان تختی)", "lat": 29.6215, "lng": 52.5412},
    {"name": "منطقه ۳ (پارسایی، سعدی، بلوار بوستان)", "lat": 29.6350, "lng": 52.5800},
    {"name": "منطقه ۴ (ستارخان، قدوسی غربی و شرقی، مطهری)", "lat": 29.6481, "lng": 52.5023},
    {"name": "منطقه ۵ (بلوار عدالت، رحمت، کلاهدوز)", "lat": 29.5750, "lng": 52.5300},
    {"name": "منطقه ۶ (معالی‌آباد انتها، تاج‌الدین غروی، دین‌دار)", "lat": 29.6850, "lng": 52.4550},
    {"name": "منطقه ۷ (بلوار مدرس، پل فسا، اتوبوس‌رانی)", "lat": 29.5821, "lng": 52.5950},
    {"name": "منطقه ۸ (هسته مرکزی - بازار وکیل، زند، لطفعلی‌خان)", "lat": 29.6104, "lng": 52.5311},
    {"name": "منطقه ۹ (بلوار استقلال، پاسداران، فضل‌آباد)", "lat": 29.6300, "lng": 52.5500},
    {"name": "منطقه ۱۰ (شهرک گلستان، صنایع، گویم)", "lat": 29.7402, "lng": 52.4112},
    {"name": "منطقه ۱۱ (شهرک مدرس، فجر، بلوار نصر)", "lat": 29.5950, "lng": 52.6100},
    {"name": "شهر جدید صدرا", "lat": 29.8050, "lng": 52.4850},
]

# ۳. دسته‌بندی جامع اصناف برای پوشش صددرصدی مشاغل
COMPREHENSIVE_CATEGORIES = [
    "پوشاک و کیف و کفش",
    "سوپرمارکت و هاپرمارکت",
    "رستوران، فست‌فود و کافه",
    "لوازم خانگی و دیجیتال",
    "خدمات خودرویی و تعمیرگاه",
    "داروخانه و تجهیزات پزشکی",
    "املاک و ساخت‌وساز",
    "خدمات آرایشی و زیبایی",
    "مصالح ساختمانی و ابزارآلات",
    "خدمات آموزشی و آموزشگاه‌ها"
]

def search_neshan_api(term, lat, lng):
    """دریافت داده از نقشه نشان بر اساس مختصات جغرافیایی منطقه"""
    url = f"https://api.neshan.org/v1/search?term={term}&lat={lat}&lng={lng}"
    headers = {"Api-Key": NESHAN_API_KEY}

    try:
        response = session.get(url, headers=headers, timeout=20)
        if response.status_code == 200:
            return response.json().get("items", [])
    except Exception as e:
        print(f"خطا در ارتباط با نقشه نشان: {e}")
    return []

def clean_data_with_gemini(raw_items, zone_name, category_name):
    """پاکسازی، دسته‌بندی و استخراج شماره‌ها با جمنای"""
    if not raw_items:
        return []

    prompt = f"""
    اطلاعات خام اماکن زیر را که مربوط به صنف '{category_name}' در محدوده '{zone_name}' در شیراز است تحلیل کن.
    خروجی را دقیقا به صورت یک آرایه JSON معتبر شامل دیکشنری‌هایی با کلیدهای زیر برگردان:
- main_street (نام خیابان اصلی به صورت کوتاه مثلا قدوسی، زند، معالی‌آباد)
- store_name (نام دقیق فروشگاه یا کسب‌وکار)
- address (آدرس کامل و دقیق)
- category (نوع کاربری دقیق و جزئی)
- phone_fixed (شماره ثابت با پیش‌شماره ۰۷۱ اگر بود وگرنه "")
- phone_mobile (شماره موبایل اگر بود وگرنه "")
- phone_second (شماره دوم اگر بود وگرنه "")

    داده‌های خام:
    {json.dumps(raw_items, ensure_ascii=False)}

    فقط و فقط JSON معتبر برگردان و هیچ متن اضافه یا علامت دیگری نگذار.
    """

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:]
        if text.endswith("```"):
            text = text[:-3]
        return json.loads(text.strip())
    except Exception as e:
        print(f"خطا در پردازش هوش مصنوعی برای {zone_name}: {e}")
        return []

def main():
    all_results = []

    print("شروع فرآیند استخراج داده‌های کل مناطق شیراز...")

    for zone in SHIRAZ_ZONES:
        for cat in COMPREHENSIVE_CATEGORIES:
            print(f"در حال استخراج صنف '{cat}' در محدوده '{zone['name']}'...")

            # دریافت داده خام از نقشه
            raw_data = search_neshan_api(cat, zone["lat"], zone["lng"])

            if raw_data:
                # پردازش و ساختاردهی با جمنای
                cleaned_items = clean_data_with_gemini(raw_data, zone["name"], cat)

                for item in cleaned_items:
                    all_results.append({
                        "محدوده جغرافیایی": item.get("main_street", zone["name"]),
                        "نام فروشگاه": item.get("store_name", ""),
                        "آدرس": item.get("address", ""),
                        "نوع کاربری": item.get("category", cat),
                        "شماره تماس ثابت": item.get("phone_fixed", ""),
                        "شماره تماس موبایل": item.get("phone_mobile", ""),
                        "شماره تماس دوم": item.get("phone_second", "")
                    })

            time.sleep(1) # وقفه استاندارد برای جلوگیری از محدودیت درخواست (Rate Limit)

    # ذخیره در فایل اکسل نهایی
    if all_results:
        df = pd.DataFrame(all_results)
        # حذف موارد کاملاً تکراری
        df.drop_duplicates(subset=["نام فروشگاه", "آدرس"], inplace=True)

        output_file = "shiraz_comprehensive_businesses.xlsx"
        df.to_excel(output_file, index=False)
        print(f"\nعملیات با موفقیت کامل انجام شد! تعداد کل رکوردهای منحصر‌به‌فرد: {len(df)}")
        print(f"فایل اکسل نهایی با نام '{output_file}' ذخیره شد.")
    else:
        print("داده‌ای یافت نشد. لطفاً کلید API نشان را بررسی کنید.")

if __name__ == "__main__":
    main()

شروع فرآیند استخراج داده‌های کل مناطق شیراز...
در حال استخراج صنف 'پوشاک و کیف و کفش' در محدوده 'منطقه ۱ (معالی‌آباد، فرهنگ‌شهر، قصردشت)'...
